# Unlimited-OCR-2 GPU canary (BlobForge evaluation)

Runs the pinned `baidu/Unlimited-OCR-2` model through its upstream Transformers path and caches the raw
generations and diagnostics to Google Drive. It produces evidence only; ConversionBundle/MDAF packaging and
scoring happen locally afterwards. See `docs/unlimited_ocr_evaluation.md` in the BlobForge repository.

**Before you start**

* Runtime → Change runtime type → **L4** or **A100** GPU. A T4 has no native BF16, so the runner refuses it.
* Upload only sources that are already cleared for hosted processing. Colab is a Google-hosted machine.
* In Drive, create `Colab Notebooks/blobforge-uocr2/` (beside your other notebooks) containing:
  * `evaluator/`: `pyproject.toml`, `uv.lock`, `run_unlimited_ocr2.py`, `plan.canary.json`
  * `inputs/`: `storypath.pdf`, `london-falling-tables.pdf`, `storypath-page-4-retained.png`

Outputs go to `Colab Notebooks/blobforge-uocr2/out/`. Completed units are never rerun, so after a disconnect you can
run every cell again and it resumes.

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total,driver_version --format=csv

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/Colab Notebooks/blobforge-uocr2'
os.environ['WORK'] = WORK
for path in ('evaluator/run_unlimited_ocr2.py', 'evaluator/uv.lock', 'evaluator/plan.canary.json', 'inputs'):
    assert os.path.exists(f'{WORK}/{path}'), f'missing {WORK}/{path}'

## Install the locked, isolated environment

The environment lives on local disk at `/content/uocr2-venv`, separate from Colab's own packages. `uv sync --frozen`
installs exactly the versions in `uv.lock` (torch 2.13.0+cu126, transformers 4.52.1, PyMuPDF 1.27.2.2).

In [ ]:
%%bash
set -euo pipefail
pip install -q uv==0.13.5
rm -rf /content/uocr2 && cp -r "$WORK/evaluator" /content/uocr2
cd /content/uocr2
UV_PROJECT_ENVIRONMENT=/content/uocr2-venv uv sync --frozen --python 3.12 --quiet
/content/uocr2-venv/bin/python -I run_unlimited_ocr2.py self-test

## Download the pinned model revision

Downloads only the files the runner pins (no bundled SGLang wheel). The runner verifies every SHA-256 before it
enables `trust_remote_code`, and it loads with `HF_HUB_OFFLINE=1`.

In [ ]:
%%bash
set -euo pipefail
/content/uocr2-venv/bin/python -I - <<'PY'
import sys
sys.path.insert(0, '/content/uocr2')
import run_unlimited_ocr2 as r
from huggingface_hub import snapshot_download
snapshot_download(r.MODEL_REPO, revision=r.MODEL_REVISION, local_dir='/content/models/Unlimited-OCR-2',
                  allow_patterns=sorted(r.MODEL_FILES))
r.verify_model(__import__('pathlib').Path('/content/models/Unlimited-OCR-2'))
print('model verified at', r.MODEL_REVISION)
PY

## 1. Timing probe: the retained v1 page

Runs the exact 300-DPI raster from the v1 CPU canary twice. Use its wall time to estimate the rest of the plan
(50 page generations in total) against your compute-unit balance before you continue.

In [ ]:
%%bash
set -euo pipefail
cd /content/uocr2
PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True /content/uocr2-venv/bin/python -I run_unlimited_ocr2.py run \
  --plan plan.canary.json --inputs "$WORK/inputs" --model-dir /content/models/Unlimited-OCR-2 \
  --out "$WORK/out" --only retained-page-local

## 2. Full canary plan

Runs in priority order: Storypath page-local, Storypath 8-page multi-page, London Falling page-local and 8-page,
Storypath 2-page chunks, then a Storypath page-local repeat. A multi-page chunk that fails its checks (truncation,
page-count mismatch, forced page break, suppressed early stop, empty page) is halved automatically. Interrupting
is safe: completed units remain cached.

In [ ]:
%%bash
set -euo pipefail
cd /content/uocr2
PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True /content/uocr2-venv/bin/python -I run_unlimited_ocr2.py run \
  --plan plan.canary.json --inputs "$WORK/inputs" --model-dir /content/models/Unlimited-OCR-2 \
  --out "$WORK/out"

In [ ]:
!/content/uocr2-venv/bin/python -I /content/uocr2/run_unlimited_ocr2.py summarize --out "$WORK/out"

## Afterwards

Download `Colab Notebooks/blobforge-uocr2/out/` into the BlobForge workspace under
`.blobforge-migration/evaluations/unlimited-ocr2-colab-canary/` and disconnect the runtime to stop spending compute units.